In [ ]:
# Based on LangChain Official Documentation
# Source: https://docs.langchain.com/oss/python/langchain/retrieval
!pip install -q pypdf
!pip install -q langchain langchain-core langchain-text-splitters
!pip install -q langchain-huggingface langchain-chroma langchain-groq
!pip install -q sentence-transformers groq
print('All libraries installed successfully!')

In [ ]:
import os
from google.colab import userdata

try:
    groq_key = userdata.get('GROQ_API_KEY')
except:
    groq_key = input("Paste your Groq API key: ").strip()

CONFIG = {
    "pdfs_path": "/content/pdfs",
    "chroma_path": "/content/chroma_db",
    # Official Docs default: chunk_size=1000, chunk_overlap=200
    # Reduced for medical text precision
    "chunk_size": 600,
    "chunk_overlap": 120,
    # Enhancement: BioBERT instead of all-mpnet (Official Docs default)
    "embedding_model": "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb",
    # Enhancement: CrossEncoder Reranker (not in Official Docs)
    "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "llm_model": "llama3-8b-8192",
    # Enhancement: retrieve 10 then rerank to top 3
    "top_k_retrieval": 10,
    "top_k_reranked": 3,
    "groq_api_key": groq_key,
    # Added batch size for ChromaDB document indexing
    "batch_size": 128
}

os.environ['GROQ_API_KEY'] = CONFIG['groq_api_key']
os.makedirs(f"{CONFIG['pdfs_path']}/esc", exist_ok=True)
os.makedirs(f"{CONFIG['pdfs_path']}/pubmed", exist_ok=True)
os.makedirs(f"{CONFIG['pdfs_path']}/arxiv", exist_ok=True)
os.makedirs(CONFIG['chroma_path'], exist_ok=True)
print('Config loaded successfully!')

In [ ]:
from google.colab import files

def upload_pdfs(folder_name):
    print(f"Upload {folder_name.upper()} files...")
    uploaded = files.upload()
    target_dir = f"{CONFIG['pdfs_path']}/{folder_name}"
    for filename, content in uploaded.items():
        with open(os.path.join(target_dir, filename), 'wb') as f:
            f.write(content)
        print(f"  Saved: {filename}")
    print(f"Done — {len(uploaded)} file(s) uploaded to {folder_name}/")

upload_pdfs('esc')
# upload_pdfs('pubmed')
# upload_pdfs('arxiv')

In [ ]:
from google.colab import files

def upload_pdfs(folder_name):
    print(f"Upload {folder_name.upper()} files...")
    uploaded = files.upload()
    target_dir = f"{CONFIG['pdfs_path']}/{folder_name}"
    for filename, content in uploaded.items():
        with open(os.path.join(target_dir, filename), 'wb') as f:
            f.write(content)
        print(f"  Saved: {filename}")
    print(f"Done — {len(uploaded)} file(s) uploaded to {folder_name}/")

# upload_pdfs('esc')
upload_pdfs('pubmed')
# upload_pdfs('arxiv')

In [ ]:
from google.colab import files

def upload_pdfs(folder_name):
    print(f"Upload {folder_name.upper()} files...")
    uploaded = files.upload()
    target_dir = f"{CONFIG['pdfs_path']}/{folder_name}"
    for filename, content in uploaded.items():
        with open(os.path.join(target_dir, filename), 'wb') as f:
            f.write(content)
        print(f"  Saved: {filename}")
    print(f"Done — {len(uploaded)} file(s) uploaded to {folder_name}/")

# upload_pdfs('esc')
# upload_pdfs('pubmed')
upload_pdfs('arxiv')

In [ ]:
import pypdf
import glob
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

def load_all_pdfs(pdfs_path):
    """
    Load all PDFs from subdirectories and attach source metadata.
    Based on LangChain Official Docs:
    https://docs.langchain.com/oss/python/langchain/knowledge-base
    Enhancement: added source_folder and real_page metadata fields.
    """
    all_docs = []
    pdf_files = glob.glob(f"{pdfs_path}/**/*.pdf", recursive=True)
    print(f"Found {len(pdf_files)} PDF file(s)\n")

    for pdf_path in pdf_files:
        try:
            reader = pypdf.PdfReader(pdf_path)
            filename = os.path.basename(pdf_path)
            source_folder = pdf_path.split('/')[-2]

            for i, page in enumerate(reader.pages):
                text = page.extract_text() or ""
                if text.strip():
                    all_docs.append(Document(
                        page_content=text,
                        metadata={
                            "source": pdf_path,
                            "filename": filename,
                            "source_folder": source_folder,
                            "page": i,
                            # PyPDF page index starts at 0, real_page corrects this
                            "real_page": i + 1,
                        }
                    ))
            print(f"  Loaded: {filename} — {len(reader.pages)} pages")
        except Exception as e:
            print(f"  Error loading {pdf_path}: {e}")

    print(f"\nTotal pages loaded: {len(all_docs)}")
    return all_docs


def split_documents(documents):
    """
    Split documents into chunks using RecursiveCharacterTextSplitter.
    Official Docs pattern with add_start_index=True.
    Chunk size reduced from 1000 to 600 for medical text precision.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CONFIG['chunk_size'],
        chunk_overlap=CONFIG['chunk_overlap'],
        add_start_index=True,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks = splitter.split_documents(documents)
    print(f"Split complete: {len(chunks)} chunks from {len(documents)} pages")
    return chunks


documents = load_all_pdfs(CONFIG['pdfs_path'])
chunks = split_documents(documents)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Official Docs pattern: HuggingFaceEmbeddings with normalize_embeddings
# Enhancement: BioBERT medical model + CUDA device
print("Loading BioBERT embeddings model...")
embeddings = HuggingFaceEmbeddings(
    model_name=CONFIG['embedding_model'],
    model_kwargs={'device': 'cuda'},
    encode_kwargs={"normalize_embeddings": True}
)

# Official Docs pattern: Chroma with persist_directory
print("\nIndexing chunks into ChromaDB...")
vector_store = Chroma(
    collection_name="cardiology_research",
    embedding_function=embeddings,
    persist_directory=CONFIG['chroma_path']
)

# Implement batching for adding documents
all_ids = []
batch_size = CONFIG['batch_size']
for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]
    print(f"  Adding batch {i//batch_size + 1}/{(len(chunks) + batch_size - 1) // batch_size} ({len(batch)} chunks)...")
    ids = vector_store.add_documents(documents=batch)
    all_ids.extend(ids)

print(f"Indexing complete — {len(all_ids)} chunks stored")

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name=CONFIG['embedding_model'],
    model_kwargs={'device': 'cuda'},
    encode_kwargs={"normalize_embeddings": True}
)

# Load existing ChromaDB instead of rebuilding
vector_store = Chroma(
    collection_name="cardiology_research",
    embedding_function=embeddings,
    persist_directory=CONFIG['chroma_path']
)
print(f"Loaded existing ChromaDB — {vector_store._collection.count()} chunks")
from sentence_transformers import CrossEncoder
from langchain_groq import ChatGroq

# Official Docs pattern: vector_store.as_retriever()
# Enhancement: top_k=10 before reranking to top 3
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": CONFIG['top_k_retrieval']}
)

# Enhancement: CrossEncoder reranker (not in Official Docs)
reranker = CrossEncoder(CONFIG['reranker_model'])
print("Retriever and reranker ready!")

llm = ChatGroq(
    model=CONFIG['llm_model'],
    temperature=0.1,
    max_tokens=1024,
    api_key=CONFIG['groq_api_key']
)
print("LLaMA-3 via Groq ready!")

In [ ]:
def expand_query(question):
    """
    Enhancement: Query Expansion (not in Official Docs).
    Generates 3 alternative phrasings to improve retrieval coverage.
    """
    prompt = f"""You are a medical research assistant.
Generate 3 different search queries for this question using different medical terminology.

Question: {question}

Return ONLY 3 queries, one per line, no numbering."""

    response = llm.invoke(prompt)
    expanded = [q.strip() for q in response.content.strip().split('\n') if q.strip()][:3]
    return [question] + expanded

print("Query expansion ready!")

In [ ]:
def retrieve_and_rerank(question, use_expansion=True):
    """
    Retrieval pipeline with optional query expansion and reranking.
    Official Docs base: retriever.invoke(query)
    Enhancement 1: Query Expansion across multiple phrasings
    Enhancement 2: CrossEncoder reranking for precision
    """
    queries = expand_query(question) if use_expansion else [question]
    print(f"  Queries generated: {len(queries)}")

    all_chunks = []
    seen = set()
    for query in queries:
        for chunk in retriever.invoke(query):
            h = hash(chunk.page_content[:100])
            if h not in seen:
                seen.add(h)
                all_chunks.append(chunk)

    print(f"  Unique chunks retrieved: {len(all_chunks)}")

    if len(all_chunks) > CONFIG['top_k_reranked']:
        scores = reranker.predict([(question, c.page_content) for c in all_chunks])
        ranked = sorted(zip(scores, all_chunks), key=lambda x: x[0], reverse=True)
        best = [c for _, c in ranked[:CONFIG['top_k_reranked']]]
        print(f"  After reranking: top {len(best)} chunks selected")
        return best

    return all_chunks

print("Retrieval pipeline ready!")

In [ ]:
SYSTEM_PROMPT = """You are a specialized cardiology research assistant.
Answer STRICTLY based on the provided research context.
Always cite sources using [Source N] notation.
If the answer is not in the context, say: 'Not available in the provided documents.'"""


def format_context(chunks):
    parts = []
    for i, c in enumerate(chunks, 1):
        parts.append(
            f"[Source {i}] {c.metadata.get('filename')} — Page {c.metadata.get('real_page')}\n{c.page_content}"
        )
    return "\n\n".join(parts)


def format_citations(chunks):
    folder_emoji = {'esc': '🏛️', 'pubmed': '🔬', 'arxiv': '⚡'}
    lines = []
    for i, c in enumerate(chunks, 1):
        emoji = folder_emoji.get(c.metadata.get('source_folder', ''), '📄')
        lines.append(
            f"  {emoji} [{i}] {c.metadata.get('filename')} — Page {c.metadata.get('real_page')}"
        )
    return "\n".join(lines)


def ask(question, use_expansion=True):
    print("=" * 60)
    print(f"Question: {question}")
    print("=" * 60)

    chunks = retrieve_and_rerank(question, use_expansion)

    prompt = f"""Based on the cardiology research documents below, answer the question.

CONTEXT:
{format_context(chunks)}

QUESTION: {question}

Provide a comprehensive answer with source citations [Source N]."""

    response = llm.invoke([("system", SYSTEM_PROMPT), ("human", prompt)])

    print("\nAnswer:")
    print("=" * 60)
    print(response.content)
    print("\nSources:")
    print("=" * 60)
    print(format_citations(chunks))


print("System ready!")

In [ ]:
# Force update CONFIG and LLM
CONFIG['llm_model'] = 'llama-3.1-8b-instant'

from langchain_groq import ChatGroq
llm = ChatGroq(
    model=CONFIG['llm_model'],
    temperature=0.1,
    max_tokens=1024,
    api_key=CONFIG['groq_api_key']
)

print(f"CONFIG model: {CONFIG['llm_model']}")
print(f"LLM model: {llm.model_name}")

In [ ]:
# Debug: find where the old model is coming from
print(f"CONFIG model: {CONFIG['llm_model']}")
print(f"LLM model: {llm.model_name}")

In [ ]:
ask("What are the diagnostic criteria for acute heart failure?")

In [ ]:
ask("What is the role of troponin in myocardial infarction diagnosis?")

In [ ]:
ask("Compare BNP and NT-proBNP as heart failure biomarkers.")

In [ ]:
print("Cardiology RAG System — type 'exit' to quit\n")
while True:
    q = input("Your question: ").strip()
    if q.lower() == 'exit':
        print("Goodbye!")
        break
    if q:
        ask(q)

In [ ]:
# Smart loader — builds only if DB doesn't exist
if os.path.exists(CONFIG['chroma_path']) and \
   os.listdir(CONFIG['chroma_path']):
    # Load existing DB
    vector_store = Chroma(
        collection_name="cardiology_research",
        embedding_function=embeddings,
        persist_directory=CONFIG['chroma_path']
    )
    print(f"Loaded existing DB — {vector_store._collection.count()} chunks")
else:
    # Build from scratch
    vector_store = Chroma.from_documents(...)
    print("Built new DB from PDFs")

In [ ]:
def evaluate_system(test_cases):
    """
    Simple self-evaluation without external metrics libraries.
    The LLM grades its own answers against expected answers.
    """
    results = []

    eval_prompt_template = """You are a medical expert evaluator.
Compare the EXPECTED answer with the ACTUAL answer and grade on 3 criteria.

Question: {question}
Expected Answer: {expected}
Actual Answer: {actual}

Grade each criterion from 1-5:
1. Accuracy: Is the information correct?
2. Completeness: Does it cover the key points?
3. Citation Quality: Are sources properly cited?

Respond in this exact format:
ACCURACY: <score>/5
COMPLETENESS: <score>/5
CITATION_QUALITY: <score>/5
OVERALL: <score>/5
VERDICT: <PASS or FAIL>"""

    print("=" * 60)
    print("SYSTEM EVALUATION REPORT")
    print("=" * 60)

    for i, test in enumerate(test_cases, 1):
        print(f"\nTest {i}/{len(test_cases)}: {test['question'][:50]}...")

        # Get system answer
        result = ask(test['question'], use_expansion=True)

        # Grade with LLM
        eval_prompt = eval_prompt_template.format(
            question=test['question'],
            expected=test['expected_answer'],
            actual=result['answer'] if isinstance(result, dict) else ""
        )

        grade = llm.invoke(eval_prompt)
        print("\nEvaluation:")
        print(grade.content)
        print("-" * 40)

        results.append({
            "question": test['question'],
            "grade": grade.content
        })

    return results


# Test cases — questions with known expected answers
test_cases = [
    {
        "question": "What are the main symptoms of acute heart failure?",
        "expected_answer": "Main symptoms include dyspnea (shortness of breath), "
                          "fatigue, orthopnea, peripheral edema, and reduced "
                          "exercise tolerance."
    },
    {
        "question": "What is the role of BNP in heart failure diagnosis?",
        "expected_answer": "BNP (B-type natriuretic peptide) is a biomarker "
                          "released by cardiac ventricles under stress. Elevated "
                          "levels confirm heart failure diagnosis and correlate "
                          "with severity."
    },
    {
        "question": "What are the ESC guidelines recommendations for "
                   "heart failure treatment?",
        "expected_answer": "ESC guidelines recommend ACE inhibitors or ARBs, "
                          "beta-blockers, and mineralocorticoid receptor "
                          "antagonists as cornerstone therapy for HFrEF."
    },
]

# Run evaluation
eval_results = evaluate_system(test_cases)